# **🚀 Vectorless RAG: PageIndex Architecture**


## 1️⃣ Install Dependencies

In [9]:
!pip install -q transformers accelerate sentencepiece pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.4/333.4 kB 22.1 MB/s eta 0:00:00


## 2️⃣ Load Hugging Face Model
We use FLAN-T5, which is strong for question answering and instruction tasks.

In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("Model loaded:", model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded: google/flan-t5-base


## 3️⃣ Creat Simple Document (Textbook style)

In [35]:
# simple small doc

document_text = """
# Introduction
Python is a high level programming language created by Guido van Rossum.

# Machine Learning
Machine learning is a field of artificial intelligence that allows systems to learn from data.

# Python Libraries
Popular Python libraries include NumPy, Pandas, and TensorFlow.

# Deep Learning
Deep learning is a subset of machine learning using neural networks.

# Data Science
Data science involves statistics, programming, and domain knowledge to analyze data.
"""

## 4️⃣ Build PageIndex (Hierarchical Index)
Instead of chunking and embeddings, we build a section index.

In [36]:
def build_page_index(text):

    index = {}
    current_section = None

    for line in text.split("\n"):

        line = line.strip()

        if line.startswith("#"):
            current_section = line.replace("#","").strip()
            index[current_section] = ""

        elif current_section:
            index[current_section] += line + " "

    return index


page_index = build_page_index(document_text)

print("PageIndex Structure:\n")

for section in page_index:
    print("-", section)

PageIndex Structure:

- Introduction
- Machine Learning
- Python Libraries
- Deep Learning
- Data Science


## 5️⃣ Vector-less Retrieval (Section Search)
This replaces vector similarity search.

In [37]:
def retrieve_section(query, index):

    query_words = query.lower().split()

    best_section_key = None
    best_section_context = "" # Initialize an empty string for context
    best_score = 0

    for section_key, section_value in index.items():
        # The searchable text initially includes the section key itself.
        current_searchable_text = section_key.lower()

        # If section_value is a dictionary, iterate through its subsections and their content
        if isinstance(section_value, dict):
            for sub_section_key, sub_section_content in section_value.items():
                current_searchable_text += " " + sub_section_key.lower()
                current_searchable_text += " " + sub_section_content.lower()
        # If the section value is directly a string (for flatter structures)
        elif isinstance(section_value, str):
            current_searchable_text += " " + section_value.lower()

        score = sum(word in current_searchable_text for word in query_words)

        if score > best_score:
            best_score = score
            best_section_key = section_key
            # Prepare the actual context to be passed to the LLM.
            if isinstance(index[best_section_key], dict):
                temp_context_parts = [best_section_key] # Always include the section key
                for sub_key, sub_content in index[best_section_key].items():
                    temp_context_parts.append(f"## {sub_key}") # Format with subsection header
                    temp_context_parts.append(sub_content)
                best_section_context = " ".join(temp_context_parts).strip()
            else: # If it's a simple string content
                best_section_context = index[best_section_key]

    print("Best Section:", best_section_key)
    print("Score:", best_score)
    return best_section_key, best_section_context

## 6️⃣ LLM Answer Generation (Hugging Face)


In [38]:
def generate_answer(query, context):

    prompt = f"""
Answer the question using the context.

Context:
{context}

Question:
{query}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    output = model.generate(
        **inputs,
        max_new_tokens=350
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

## 7️⃣ Run the Vector-less RAG Pipeline

In [40]:
query = "Who created Python"

section, context = retrieve_section(query, page_index)

print("Retrieved Section:", section)
print("\nContext:\n", context)

answer = generate_answer(query, context)

print("\nFinal Answer:\n", answer)

Best Section: Introduction
Score: 2
Retrieved Section: Introduction

Context:
 Python is a high level programming language created by Guido van Rossum.  

Final Answer:
 Guido van Rossum
